# CellNeighborEX v2 Tutorial — 05. Identifying Spatial Region-Specific CCI Genes in Human Colorectal Cancer (CRC)

**Key concept:** Colorectal cancer contains spatially heterogeneous regions, including the tumor microenvironment (TME), which allow evaluation of whether *spatial region–specific CCI genes* are associated with functional pathway activity.

This notebook starts from bundled **precomputed `ccisignal` outputs**. We therefore skip the deconvolution step and focus on identifying spatial region-specific CCI genes and their interacting cell type pairs using the downstream modules `ccigenes` and `ccipairs`.

## Core Concept: Interpreting Spatial CCI Signals in Colorectal Cancer

**Colorectal cancer contains spatially heterogeneous regions, including the tumor microenvironment (TME). This tutorial focuses on the discovery of spatial region-specific CCI genes and their interacting cell type pairs.**

### Using human colorectal cancer spatial transcriptomics data as an example, this tutorial demonstrates how to:

- Analyze spatial cell type distributions across tumor-associated regions
- Identify spatial region–specific CCI genes 
- Infer interacting cell type pairs for each CCI gene

## ⚙️ User Setup

Before running this notebook, set `RUN_CODE_DIR` in the cell below to the path of your local `CNEXv2/run_code` directory.

**Example:**
- Linux / Mac: `Path("/home/username/CNEXv2/run_code")`
- Windows: `Path("C:/Users/username/CNEXv2/run_code")`


In [ ]:
from pathlib import Path

# ============================================================
#  USER CONFIGURATION — Edit before running
# ============================================================
RUN_CODE_DIR = Path("/path/to/CNEXv2/run_code").resolve()  # ← Change to your path
# ============================================================


In [ ]:
import os
import sys

# --- Environment Configuration ---
# Set thread limits for linear algebra and numerical libraries to ensure 
# computational efficiency and stability across different environments.
os.environ.setdefault("MKL_THREADING_LAYER", "SEQUENTIAL")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "4")

# --- Standard Library Imports ---
import numpy as np
import pandas as pd
import scanpy as sc
from IPython.display import display

# --- Path Setup ---
# Set the project base directory (parent of the run_code folder).
BASE_DIR = RUN_CODE_DIR.parent  # CNEXv2

# Add the project base directory to the system path to allow local imports of CellNeighborEX.
sys.path.insert(0, str(BASE_DIR))

# --- CellNeighborEX Module Imports ---

# 1. ccisignal: Functions for spatial signal processing and cell-type proportion analysis.
from CellNeighborEX.ccisignal import compute_proportion

# 2. ccigenes: Statistical tools for identifying and validating CCI-associated genes.
from CellNeighborEX.ccigenes import (
    clean_column_names,
    obtain_clean_celltype_names,
    permutation_test_all_clusters,
    chi_square_goodness_of_fit,
    compute_combined_p_value,
    simplify_gene_names,
)

# 3. ccipairs: Regression-based modeling to identify specific interacting cell-type pairs.
from CellNeighborEX.ccipairs import (
    regress_residual_on_interaction_with_regularization,
    extract_interaction_terms,
    compare_database,
)

/home/Data_Drive_8TB/sglee/.conda/envs/cnex2_env/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.0' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/Data_Drive_8TB/sglee/.conda/envs/cnex2_env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/Data_Drive_8TB/sglee/.conda/envs/cnex2_env/lib/python3.9/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/home/Data_Drive_8TB/sglee/.conda/envs/cnex2_env/lib/python3.9/site-packages/scvi/_settings.py:63: UserWarning: Since v1.0.0, scvi-tools no longer uses a random seed by default. Run `scvi.settin

In [ ]:
# --- Data Directory Configuration (Colorectal Cancer) ---
# Define the specific tutorial path for the CRC dataset
DATA_DIR = BASE_DIR / "Tutorial-ExampleData" / "5_colorectal_cancer"
# Single-cell RNA-seq reference data used for cell type signatures
SC_REF_FILE = DATA_DIR / "sc_ccisignal.h5ad"
# Spatial transcriptomics (Visium) data to be analyzed
VISIUM_FILE = DATA_DIR / "sp_ccisignal.h5ad"

# --- Analysis Metadata ---
# Target species for biological database mapping (human-specific L-R pairs)
SPECIES = "human"
# Define the spatial grouping key; 'proportion_leiden1' is the default niche for this data
CLUSTER_INFO = "proportion_leiden1"

# --- Analysis Hyperparameters (Optimization & Significance) ---
# Set the number of iterations for the permutation test (200 for CRC tutorial)
N_PERMUTATIONS = 200
# Define the minimum Log Fold Change threshold for CCI gene filtering
LOG_FC = 0.5
# Set the p-value significance threshold for extracting interaction terms
PVAL_TERM = 0.05

# --- Output Directory Management ---
# Create the root output directory for CRC results
OUTPUT_DIR = RUN_CODE_DIR / "tutorial_output" / "05_colorectal_cancer"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Create a subdirectory specifically for saving visualization figures
(OUTPUT_DIR / "figures").mkdir(exist_ok=True)

# --- Visualization Settings ---
# Configure Scanpy's figure saving path to the newly created directory
sc.settings.figdir = str((OUTPUT_DIR / "figures").resolve())
# Set global figure resolution (DPI) for clear visual output
sc.settings.set_figure_params(dpi=120)

# --- Configuration Verification ---
# Print the active file paths to the console to verify setup
print(f"SC_REF_FILE : {SC_REF_FILE}")
print(f"VISIUM_FILE : {VISIUM_FILE}")
print(f"OUTPUT_DIR  : {OUTPUT_DIR}")

SC_REF_FILE : /home/Data_Drive_8TB/sglee/Project/01.Neighbor/drive-download-20251001T080437Z-1-001/CNEXv2/Tutorial-ExampleData/5_colorectal_cancer/sc_ccisignal.h5ad
VISIUM_FILE : /home/Data_Drive_8TB/sglee/Project/01.Neighbor/drive-download-20251001T080437Z-1-001/CNEXv2/Tutorial-ExampleData/5_colorectal_cancer/sp_ccisignal.h5ad
OUTPUT_DIR  : /home/Data_Drive_8TB/sglee/Project/01.Neighbor/drive-download-20251001T080437Z-1-001/CNEXv2/run_code/tutorial_output/05_colorectal_cancer


## 1) Load data and check meta information

The following datasets are loaded:
- `sc_ccisignal.h5ad`: reference signatures
- `sp_ccisignal.h5ad`: spatial deconvolution map

In [ ]:
# --- Data Loading ---
# Load the single-cell reference signature and the spatial transcriptomics dataset
adata_ref_sig = sc.read_h5ad(SC_REF_FILE)
adata_vis = sc.read_h5ad(VISIUM_FILE)

# Display a high-level summary of the AnnData objects
print("Reference:", adata_ref_sig)
print("Spatial:  ", adata_vis)

# --- Integrity Checks ---
# Ensure 'mod' exists in .uns for both datasets (contains deconvolution/model parameters)
assert "mod" in adata_ref_sig.uns and "mod" in adata_vis.uns

# Confirm that the cell abundance estimates (from deconvolution) are present in .obsm
assert "q05_cell_abundance_w_sf" in adata_vis.obsm

# --- Sample Identity Assignment ---
# If a 'sample' column is missing, extract the first key from .uns['spatial'] 
# or default to 'sample0' to ensure compatibility with downstream functions
if "sample" not in adata_vis.obs.columns and "spatial" in adata_vis.uns:
    adata_vis.obs["sample"] = list(adata_vis.uns["spatial"].keys())[0]
elif "sample" not in adata_vis.obs.columns:
    adata_vis.obs["sample"] = "sample0"

# --- Cluster Column Validation ---
# Verify that the specified CLUSTER_INFO column exists in the metadata (.obs)
if CLUSTER_INFO not in adata_vis.obs.columns:
    raise ValueError(
        f"CLUSTER_INFO='{CLUSTER_INFO}' not in adata_vis.obs. "
        f"Candidates: {[c for c in adata_vis.obs.columns if 'leiden' in c or c == 'cluster'][:20]}"
    )

# Preview the specific niche/cluster assignments for the first few spots
display(adata_vis.obs[[CLUSTER_INFO]].head())

Reference: AnnData object with n_obs × n_vars = 37000 × 11964
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'Patient', 'BC', 'Barcode', 'UMAP.1', 'UMAP.2', 'MT.percent', 'QCFilter', 'leverage.score', 'sketch_snn_res.0.6', 'seurat_clusters', 'Level1', 'Level2', 'ProjectAll_L1', 'ProjectAll_L1.score', 'ProjectAll_L2', 'ProjectAll_L2.score', '_indices', '_scvi_batch', '_scvi_labels'
    var: 'features', 'SYMBOL', 'n_cells', 'nonz_mean'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'mod'
    varm: 'means_per_cluster_mu_fg', 'q05_per_cluster_mu_fg', 'q95_per_cluster_mu_fg', 'stds_per_cluster_mu_fg'
Spatial:   AnnData object with n_obs × n_vars = 4269 × 11746
    obs: 'in_tissue', 'array_row', 'array_col', 'sample', 'n_genes_by_counts', 'total_counts', '_indices', '_scvi_batch', '_scvi_labels', 'Adipocyte', 'CAF', 'CD4 T cell', 'CD8 Cytotoxic T cell', 'Endothelial', 'Enteric Glial', 'Enterocyte', 'Epithelial', 'Fibroblast', 'Goblet', 'Lymphatic Endothelial', 'Macrophage', 'Mast', 'Matu

,proportion_leiden1
AACAATGTGCTCCGAG-1,0
AACACCATTCGCATAC-1,2
AACACGACAACGGAGT-1,1
AACACGCAGATAACAA-1,2
AACACGTTGATACCGC-1,0


## 2) Build observed/expected matrices 

Upon completing the `ccisignal` module, you typically obtain two primary output files: `sc_ccisignal.h5ad` and `sp_ccisignal.h5ad`. For this tutorial, these two files are provided as precomputed outputs to allow you to proceed directly with the analysis. Using these datasets, we calculate the expected expression values to compare them against the observed expression data.

In [ ]:
# --- Configuration and Gene Name Normalization ---
cluster_info = CLUSTER_INFO

# Standardize gene identifiers based on the target species
simplify_gene_names(adata_ref_sig, SPECIES)
simplify_gene_names(adata_vis, SPECIES)

# --- Reference Signature Extraction (Gene × Cell Type) ---
# Identify the cell type factors used in the single-cell reference model
factor_names = list(adata_ref_sig.uns["mod"]["factor_names"])
# Retrieve the mean expression matrix (mu_fg) from the reference signatures
inf_raw = adata_ref_sig.varm.get("means_per_cluster_mu_fg", None)

if inf_raw is None:
    inf_aver2 = adata_ref_sig.obs[factor_names].copy()
elif isinstance(inf_raw, pd.DataFrame):
    inf_aver2 = inf_raw.copy()
else:
    inf_aver2 = pd.DataFrame(inf_raw, index=adata_ref_sig.var_names, columns=factor_names)

# Clean column names by removing the technical prefix if present
prefix = "means_per_cluster_mu_fg_"
if all(isinstance(c, str) and c.startswith(prefix) for c in inf_aver2.columns):
    inf_aver2.columns = [c[len(prefix):] for c in inf_aver2.columns]

# --- Feature and Cell Type Alignment ---
# Intersect genes to ensure consistency between spatial data and reference signature
genes = np.intersect1d(adata_vis.var_names, inf_aver2.index)
adata_vis = adata_vis[:, genes].copy()
inf_aver2 = inf_aver2.loc[genes]

# Align cell type factors between the spatial deconvolution output and reference
vis_factors = list(adata_vis.uns["mod"]["factor_names"])
common = [ct for ct in vis_factors if ct in inf_aver2.columns]
if not common:
    raise ValueError(f"No overlapping celltypes. vis={vis_factors}, ref={list(inf_aver2.columns)}")

# Map cell abundance estimates from .obsm to .obs if not already present
if not all(ct in adata_vis.obs.columns for ct in common):
    adata_vis.obs[vis_factors] = pd.DataFrame(
        adata_vis.obsm["q05_cell_abundance_w_sf"], index=adata_vis.obs_names, columns=vis_factors
    )

# --- Reconstructing Expected Expression ---
# Use posterior means to calculate the baseline expression expected from cell composition
post = adata_vis.uns["mod"]["post_sample_means"]

# Multiply Cell Proportions (Spot x CT) by Reference Signatures (CT x Gene)
total_df = adata_vis.obs[common] @ inf_aver2[common].T

# Incorporate technical scaling: m_g (sensitivity), s_g (background), and y_s (detection efficiency)
final_df = (total_df.mul(post["m_g"], axis=1) + np.asarray(post["s_g_gene_add"])[0]).mul(post["detection_y_s"], axis=0)

# --- Dataset Integration for Statistical Testing ---
meta_data = adata_vis.obs.copy()
# Create paired matrices for Observed and Expected expression values
observed_expression = pd.concat([adata_vis.to_df(), meta_data], axis=1)
expected_expression = pd.concat([final_df, meta_data], axis=1)

# Annotate conditions and modify indices to distinguish the two groups
observed_expression["condition"] = "observed"
expected_expression["condition"] = "expected"
observed_expression.index = [f"{i}_obs" for i in observed_expression.index]
expected_expression.index = [f"{i}_exp" for i in expected_expression.index]

# Concatenate datasets and build a new AnnData object for differential testing
combined_df = pd.concat([observed_expression, expected_expression])
drop_cols = list(meta_data.columns) + ["condition"]
adata_tests = sc.AnnData(X=combined_df.drop(columns=drop_cols))
adata_tests.obs["condition"] = combined_df["condition"].values
adata_tests.obs[cluster_info] = combined_df[cluster_info].astype("category")

# --- Final Sanitization ---
# Clean all column names and obtain finalized cell type identifiers
observed_expression = clean_column_names(observed_expression)
expected_expression = clean_column_names(expected_expression)
meta_data = clean_column_names(meta_data)
inf_aver2 = clean_column_names(inf_aver2)
cell_types = obtain_clean_celltype_names(adata_vis)

print("OK:", observed_expression.shape, expected_expression.shape, "celltypes used:", len(common))

OK: (4269, 11802) (4269, 11802) celltypes used: 39


## 3) Detect spatial region-specific CCI genes and interacting cell-type pairs (`ccigenes` & `ccipairs` modules)

To identify genes driven by cell-cell interactions (CCI), we employ a robust statistical framework that integrates two distinct approaches: a `permutation test` to evaluate expression differences across spatial regions and a `chi-square test` to assess variance within each region. The results from both tests are combined using the `Cauchy combination test` to determine overall statistical significance. Final CCI gene candidates are then filtered based on adjusted `p-values` and `log-fold change` (LogFC) thresholds.

Note: This analytical pipeline is consistent across various datasets; only the `SPECIES` and `CLUSTER_INFO` parameters require adjustment to match the specific sample metadata.

Following the identification of CCI genes, we determine the specific interacting cell-type pairs through a `regression-based framework` (a two-step modeling strategy). In this step, we regress the residual expression signals against cell-type interaction terms to quantify pairwise effects. Finally, these significant interaction terms are biologically annotated by mapping them to curated ligand-receptor databases (`Omnipath`, `CellChat`, `CellTalk`)., bridging the gap between observed spatial signals and established molecular signaling pathways.

In [ ]:
# --- Identify CCI Genes (Niche Genes) via Statistical Testing ---

# Run a permutation test to compare observed expression against composition-based expected expression
perm_results = permutation_test_all_clusters(
    adata_tests,
    cluster_info=cluster_info,
    observed_expression=observed_expression,
    expected_expression=expected_expression,
    n_permutations=N_PERMUTATIONS,
    use_zeros=True,
    random_seed=42,
    path=str(OUTPUT_DIR),
)

# Perform a Chi-square goodness-of-fit test to validate the variance between conditions
chi_results = chi_square_goodness_of_fit(
    adata_tests,
    cluster_info=cluster_info,
    groupby="condition",
    reference="observed",
    target="expected",
    use_zeros=True,
)

# Merge statistical results and calculate a combined adjusted p-value using the Cauchy method
stats_df = pd.merge(perm_results, chi_results, on=["gene", "cluster"], how="inner")
stats_df["combined_p_value_adj"] = stats_df.apply(compute_combined_p_value, axis=1)
stats_df.to_csv(OUTPUT_DIR / "allgenes_results.csv", index=False)

# Filter for significant CCI genes: low p-value, higher observed mean, and sufficient Log Fold Change
gene_filter = (
    (stats_df["combined_p_value_adj"] < 0.01)
    & (stats_df["mean_ref"] > stats_df["mean_tgt"])
    & (stats_df["logfc"] > LOG_FC)
)
final_results = stats_df.loc[gene_filter].copy()
final_results.to_csv(OUTPUT_DIR / "ccigenes_results.csv", index=False)

print("Niche gene hits:", final_results.shape[0])
display(final_results[["gene", "cluster", "combined_p_value_adj", "logfc"]].head(20))

# --- Infer Interacting Cell-Type Pairs for Identified CCI Genes ---

interaction_terms = pd.DataFrame()
if final_results.empty:
    print("No niche genes detected; skipping ccipairs.")
else:
    # Normalize cell-type deconvolution data to get spot-level proportions
    norm_deconv = meta_data.loc[:, cell_types].div(meta_data.loc[:, cell_types].sum(axis=1), axis=0)
    norm_deconv[cluster_info] = meta_data[cluster_info]
    
    # Calculate mean cell-type composition for each spatial cluster
    cluster_summary = norm_deconv.groupby(cluster_info).mean()
    cluster_summary.loc["mean"] = cluster_summary.mean()

    # Clean data by removing duplicate gene entries
    inf_aver2 = inf_aver2[~inf_aver2.index.duplicated(keep="first")]
    final_results = final_results.drop_duplicates(subset=["gene"])

    # Model the residual signal as a function of cell-type interactions using regularized regression
    models_per_gene, gene_analysis = regress_residual_on_interaction_with_regularization(
        observed_expression=observed_expression,
        expected_expression=expected_expression,
        celltypes=cell_types,
        cell_sig=inf_aver2,
        niche_gene_results=final_results,
        cluster_summary=cluster_summary,
        cluster_info=cluster_info,
        self_interaction=False,
        use_zeros=False,
        alpha=0.5,
    )

    # Extract cell-type pairs that significantly drive the interaction signal
    interaction_terms = extract_interaction_terms(models_per_gene, gene_analysis, p_value_threshold=PVAL_TERM)
    
    if interaction_terms.empty:
        print("No significant interaction terms.")
    else:
        # Keep the top 5 most significant cell-type pairs per gene per cluster
        interaction_terms = (
            interaction_terms.groupby(["cluster", "gene"]) 
            .apply(lambda df: df.sort_values("p_value").head(5))
            .reset_index(drop=True)
        )
        
        # Cross-reference findings with known Ligand-Receptor databases
        interaction_terms = compare_database(interaction_terms, species=SPECIES)
        
        # Save validated interaction results
        interaction_terms.to_csv(OUTPUT_DIR / "ccipairs_results.csv", index=False)
        display(interaction_terms.head(25))

Permutation Test Progress for Cluster 6: 100%|██████████| 11746/11746 [00:46<00:00, 253.42it/s]


Performing Peason's Chi-Square Test for cluster 0...


Chi-Square Test Progress: 100%|██████████| 11746/11746 [00:13<00:00, 865.33it/s]


Performing Peason's Chi-Square Test for cluster 1...


Chi-Square Test Progress: 100%|██████████| 11746/11746 [00:12<00:00, 925.13it/s]


...

Skipping single regression for ST3GAL1, variable CAF_Tumor_IV, cluster 6: NaN, inf or invalid value detected in weights, estimation infeasible.
Skipping single regression for ST3GAL1, variable CD8_Cytotoxic_T_cell_Vascular_Fibroblast, cluster 6: NaN, inf or invalid value detected in weights, estimation infeasible.
Skipping single regression for ST3GAL1, variable Adipocyte_Goblet, cluster 6: NaN, inf or invalid value detected in weights, estimation infeasible.
Skipping single regression for ST3GAL1, variable Proliferating_Macrophage_Tumor_IV, cluster 6: NaN, inf or invalid value detected in weights, estimation infeasible.
Skipping single regression for ST3GAL1, variable Endothelial_Tumor_IV, cluster 6: NaN, inf or invalid value detected in weights, estimation infeasible.
Skipping single regression for ST3GAL1, variable Smooth_Muscle_Tumor_IV, cluster 6: NaN, inf or invalid value detected in weights, estimation infeasible.
Skipping single regression for ST3GAL1, variable CD8_Cytotoxic_T_

Processing Genes: 100%|██████████| 1169/1169 [39:22<00:00,  2.02s/gene]


Skipping IFIT1 in cluster 5 due to missing models.
Skipping KRT23 in cluster 5 due to missing models.
Skipping LGR5 in cluster 5 due to missing models.
Skipping MX1 in cluster 5 due to missing models.
Skipping NKD1 in cluster 5 due to missing models.
Skipping RBP4 in cluster 5 due to missing models.
Skipping RSAD2 in cluster 5 due to missing models.
Skipping ADAM28 in cluster 6 due to missing models.
Skipping AKR1B1 in cluster 6 due to missing models.
Skipping CYTIP in cluster 6 due to missing models.
Skipping SCARA5 in cluster 6 due to missing models.
Skipping ST3GAL1 in cluster 6 due to missing models.


Processing Queries: 100%|██████████| 3320/3320 [01:33<00:00, 35.32it/s]


,gene,cluster,term,index_celltype,neighboring_celltype,coef,std_err,p_value,source_omnipath,pair_omnipath,source_cellchat,pair_cellchat,source_celltalk,pair_celltalk
0,APOC2,0,Fibroblast_Macrophage,Macrophage,Fibroblast,0.358752,0.106196,7.295910e-04,omnipath,single HNF4A-APOC2; single APOC2-LDLR; single ...,unknown,unknown,celltalk,single APOC2-LDLR; single APOC2-LRP1; single A...
1,APOC2,0,Adipocyte_Macrophage,Macrophage,Adipocyte,0.018296,0.005491,8.626198e-04,omnipath,single HNF4A-APOC2; single APOC2-LDLR; single ...,unknown,unknown,celltalk,single APOC2-LDLR; single APOC2-LRP1; single A...
2,ARHGAP10,0,CAF_Macrophage,Macrophage,CAF,0.006254,0.001569,6.743993e-05,omnipath,single ARHGAP10-RHOA,unknown,unknown,unknown,unknown
3,ARHGAP10,0,Macrophage_Unknwon_I_Immune,Macrophage,Unknwon_I_Immune,0.052494,0.017633,2.910614e-03,omnipath,single ARHGAP10-RHOA,unknown,unknown,unknown,unknown
4,ARHGAP10,0,Macrophage_Plasma,Macrophage,Plasma,0.067361,0.025021,7.098812e-03,omnipath,single ARHGAP10-RHOA,unknown,unknown,unknown,unknown
5,ATF3,0,CAF_mRegDC,mRegDC,CAF,0.296404,0.042523,3.160556e-12,omnipath,single TP53-ATF3; single ATF3-TP53; single ATF...,unknown,unknown,unknown,unknown
6,ATF3,0,Neuroendocrine_mRegDC,mRegDC,Neuroendocrine,3.174972,0.492501,1.143632e-10,omnipath,single TP53-ATF3; single ATF3-TP53; single ATF...,unknown,unknown,unknown,unknown
7,ATF3,0,Unknwon_I_Immune_mRegDC,mRegDC,Unknwon_I_Immune,3.791898,0.639517,3.041964e-09,omnipath,single TP53-ATF3; single ATF3-TP53; single ATF...,unknown,unknown,unknown,unknown
8,ATF3,0,Proliferating_Macrophage_mRegDC,mRegDC,Proliferating_Macrophage,17.665526,3.428745,2.574692e-07,omnipath,single TP53-ATF3; single ATF3-TP53; single ATF...,unknown,unknown,unknown,unknown
9,ATF3,0,CAF_Macrophage,Macrophage,CAF,0.006801,0.001450,2.740401e-06,omnipath,single TP53-ATF3; single ATF3-TP53; single ATF...,unknown,unknown,unknown,unknown
